In [ ]:
import kagglehub
import pandas as pd
import os
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, recall_score, precision_score, f1_score
from xgboost import XGBClassifier
from sklearn.inspection import permutation_importance

path = kagglehub.dataset_download(
    "blastchar/telco-customer-churn"
)

csv_path = os.path.join(
    path,
    "WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

df = pd.read_csv(csv_path)

#print(df.describe())

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()

categorical_cols = df.select_dtypes(include=['object','str']).columns
categorical_cols = categorical_cols.drop('customerID')
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
# print(categorical_cols)

#print(df_encoded.describe())

x_raw = df_encoded.drop('Churn_Yes', axis=1)
x_raw = x_raw.drop('customerID', axis=1)
x_scaled = StandardScaler().fit_transform(x_raw)
pca = PCA(n_components = 2)
x_pca = pca.fit_transform(x_scaled)
y = df['Churn'].map({'No': 0, 'Yes': 1})
X = x_pca


kmeans = KMeans(n_clusters=5, init='k-means++', random_state=12)
y_kmeans = kmeans.fit_predict(X)


#plt.scatter(X[:, 0], X[:, 1], c=y_kmeans, s=50, cmap='viridis')
#plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], s=200, c='red', marker='X')
#plt.show()

#model function
def build_model(x_raw,y, method):
    x_train, x_test, y_train, y_test = train_test_split(x_raw, y, test_size=0.2, random_state=123, stratify=y)

    model = method #RandomForestClassifier(random_state=92)
    #print(x_train)
    model.fit(x_train, y_train)

    predictions = model.predict(x_test)
    print(method)
    accuracy = accuracy_score(y_test, predictions)

    print(f"Model Accuracy: {accuracy * 100:.2f}%")
    
    
    print(f"Classification Report:\n {classification_report(y_test, predictions)}")
    
    probs = model.predict_proba(x_test)[:,1]
    #return()
    if isinstance(model, LogisticRegression):
        importance = pd.Series(
            model.coef_[0],
            index=x_train.columns
        ).sort_values()
        #print(importance)
    elif hasattr(model, "feature_importances_"):
        importance = pd.Series(
            model.feature_importances_,
            index=x_train.columns
        ).sort_values(ascending=False)
        #print(importance)
    
    result = permutation_importance(
        model,
        x_test,
        y_test,
        n_repeats=10,
        random_state=123
    )
    
    importance = pd.Series(
        result.importances_mean,
        index=x_test.columns
    ).sort_values(ascending=False)
    
    return {
    "model": str(model),
    "accuracy": accuracy_score(y_test, predictions),
    "precision": precision_score(y_test, predictions),
    "recall": recall_score(y_test, predictions),
    "f1": f1_score(y_test, predictions),
    "auc": roc_auc_score(y_test, probs)
    }

    #print(importance.head(10))
 

#Model 1
results = []

results.append(build_model(x_raw,y,LogisticRegression(max_iter=5000)))
results.append(build_model(x_raw,y,RandomForestClassifier()))
results.append(build_model(x_raw,y,XGBClassifier()))

#Model 2
build_model(x_raw,y,LogisticRegression(max_iter=5000,l1_ratio=0))
build_model(x_raw,y,RandomForestClassifier(n_estimators=500,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42))
build_model(x_raw,y,XGBClassifier(max_depth=4,
    learning_rate=0.05,
    n_estimators=500))

results_df = pd.DataFrame(results)
display(results_df.sort_values("auc", ascending=False))


LogisticRegression(max_iter=5000)
Model Accuracy: 80.03%
Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.90      0.87      1033
           1       0.65      0.53      0.59       374

    accuracy                           0.80      1407
   macro avg       0.75      0.71      0.73      1407
weighted avg       0.79      0.80      0.79      1407

RandomForestClassifier()
Model Accuracy: 78.25%
Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.89      0.86      1033
           1       0.62      0.48      0.54       374

    accuracy                           0.78      1407
   macro avg       0.72      0.69      0.70      1407
weighted avg       0.77      0.78      0.77      1407

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds

,model,accuracy,precision,recall,f1,auc
0,LogisticRegression(max_iter=5000),0.800284,0.652459,0.532086,0.586156,0.840791
2,"XGBClassifier(base_score=None, booster=None, c...",0.779673,0.600000,0.513369,0.553314,0.810097
1,RandomForestClassifier(),0.782516,0.615646,0.483957,0.541916,0.808616
